In [1]:
import mysql.connector
import pandas as pd
import warnings
from pandasql import sqldf
import modules as mod
import numpy as np
warnings.filterwarnings('ignore')

dbCredentials = mod.getSecret()
cnx = mysql.connector.connect(
    user=dbCredentials['username'],
    password=dbCredentials['password'],
    host=dbCredentials['host'],
    database=dbCredentials['dbInstanceIdentifier']
)
cursor = cnx.cursor()

print('We connected with secrets manager!')

We connected with secrets manager!


In [2]:
query = """
SELECT *
from fighterHyperlinks
"""
cursor.execute(query)
result = cursor.fetchall()
column_names = [column[0] for column in cursor.description]
fightersTable = pd.DataFrame(result, columns=column_names)

query = """
SELECT fs.*, ehl.eventDate
from fightStats fs LEFT JOIN
eventHyperlinks ehl ON ehl.eventID = fs.eventID
"""
cursor.execute(query)
result = cursor.fetchall()
column_names = [column[0] for column in cursor.description]
fightsTable = pd.DataFrame(result, columns=column_names)


In [3]:
"""
This block is used for mainly clean up and prep on fightsTable so that we can create all the metrics that we need later on
"""
fightsTable['round'] = fightsTable['round'].str.replace('Round:', '')
fightsTable['time'] = fightsTable['time'].str.replace('Time:', '')

# Split time columns and find actual length of bout in minutes
fightsTable[['minutes', 'seconds']] = fightsTable['time'].str.split(':', expand=True)
fightsTable['minutes'] = fightsTable['minutes'].astype(int)
fightsTable['seconds'] = fightsTable['seconds'].astype(int)
fightsTable['round'] = fightsTable['round'].astype(int)
fightsTable['Minutes In Fight'] = ((fightsTable['round'] - 1) * 5) + fightsTable['minutes'] + fightsTable['seconds'] / 60
fightsTable.drop(['minutes', 'seconds'], axis=1, inplace=True)
fightsTable['eventDate'] = pd.to_datetime(fightsTable['eventDate'])
fightsTable['fighter_A_sig_strikes'] = fightsTable['fighter_A_sig_strikes'].str.split(' ').str[0].astype(int)
fightsTable['fighter_B_sig_strikes'] = fightsTable['fighter_B_sig_strikes'].str.split(' ').str[0].astype(int)
fightsTable['fighter_A_takedowns'] = fightsTable['fighter_A_takedowns'].str.split(' ').str[0].astype(int)
fightsTable['fighter_B_takedowns'] = fightsTable['fighter_B_takedowns'].str.split(' ').str[0].astype(int)


fightersTable['Strikes_Landed_Per_Minute'] = fightersTable['Strikes_Landed_Per_Minute'].astype(float)
fightersTable['Strikes_Absorbed_Per_Minute'] = fightersTable['Strikes_Absorbed_Per_Minute'].astype(float)
fightersTable['Takedown_Average'] = fightersTable['Takedown_Average'].astype(float)
fightersTable['Submission_Average'] = fightersTable['Submission_Average'].astype(float)
fightersTable['DOB'] = fightersTable['DOB'].replace({'--': np.nan})
fightersTable['DOB'] = pd.to_datetime(fightersTable['DOB'])

In [4]:
###
#UPDATED -> dont include current row when calculating win percentage
###
""" 
This query allows for us to see the win percentage of fighters throughout their careers
"""
query = """ 
SELECT fighterID, fighter, eventDate, eventID, fightID, AVG(winCount) OVER (PARTITION BY fighterID ORDER BY eventDate ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) as winPercentage
FROM(
    SELECT fighterA as fighter, fighter_A_ID as fighterID, winner, eventDate, eventID, fightID,
    CASE
        WHEN winner = fighterA THEN 1
        ELSE 0
    END as winCount
    FROM fightsTable 
    UNION ALL
    SELECT fighterB as fighter, fighter_B_ID as fighterID, winner, eventDate, eventID, fightID,
    CASE
        WHEN winner = fighterB THEN 1
        ELSE 0
    END as winCount
    FROM fightsTable
) as winCounts
ORDER BY fighterID, eventDate
"""
winPercentageResult = sqldf(query)

In [5]:
###
#UPDATED -> we now dont include current row when calculating average fight time
###
""" 
This table is dedicated to calculating the average fight time for a fighter throughout their ufc career
"""

query = """
SELECT fighter, fighterID, eventDate, eventID, fightID, AVG(`Minutes In Fight`) OVER (PARTITION BY fighterID ORDER BY eventDate ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) as `Average Fight Time`
FROM(
    SELECT fighterA as fighter, fighter_A_ID as fighterID, `Minutes In Fight`, eventDate, eventID, fightID
    FROM fightsTable
    UNION ALL
    SELECT fighterB as fighter, fighter_B_ID as fighterID, `Minutes In Fight`, eventDate, eventID, fightID
    FROM fightsTable
) innerQuery
order by fighterID, eventDate
"""
averageFightTimeResult = sqldf(query)

In [6]:
###
#UPDATED -> we dont have to accomandate the DOB since going in they'll be same age as going out
###
""" 
This block shows us the age of the fighter at the time of the bout
"""
query = """
select fighter, innerQuery.fighterID, DOB, eventID, fightID, 
       (julianday(innerQuery.eventDate) - julianday(DOB)) / 365.25 as Age
from(
    SELECT fighterA as fighter, fighter_A_ID as fighterID, `Minutes In Fight`, eventDate, eventID, fightID
    FROM fightsTable
    UNION ALL
    SELECT fighterB as fighter, fighter_B_ID as fighterID, `Minutes In Fight`, eventDate, eventID, fightID
    FROM fightsTable
)innerQuery
LEFT JOIN fightersTable on innerQuery.fighterID = fightersTable.fighterID
"""
ageResult = sqldf(query)

In [7]:
###
#UPDATED -> when calculating fight experience we will not include current row
###
""" 
This code block is dedicated towards finding the # of ufc fights (experience) a fighter had going into a bout
"""

query = """ 
select fighter, fighterID, eventDate, eventID, count(*) OVER (PARTITION BY fighterID ORDER BY eventDate ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) as `Number of Fights`
from(
    SELECT fighterA as fighter, fighter_A_ID as fighterID, `Minutes In Fight`, eventDate, eventID, fightID
    FROM fightsTable
    UNION ALL
    SELECT fighterB as fighter, fighter_B_ID as fighterID, `Minutes In Fight`, eventDate, eventID, fightID
    FROM fightsTable
) innerQuery
"""
experienceResult = sqldf(query)

In [8]:
###
#UPDATED -> when calculating finish rate we no longer include the current row
###
""" 
This block is focused on finding the finish rate for fighters among their wins
"""

query = """ 
select fighter, fighterID, eventDate, eventID, fightID, AVG(numMethod) OVER (PARTITION BY fighterID ORDER BY eventDate ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) as `Finish Rate`
FROM(
    SELECT fighterA as fighter, fighter_A_ID as fighterID, eventDate, eventID, fightID,
    CASE
        WHEN method = 'KO/TKO' THEN 1
        WHEN method = 'Submission' THEN 1
        WHEN method = 'TKO - Doctor''s Stoppage' THEN 1
        ELSE 0
    END AS numMethod
    FROM fightsTable
    where winner = fighter
    UNION ALL
    SELECT fighterB as fighter, fighter_B_ID as fighterID, eventDate, eventID, fightID,
    CASE
        WHEN method = 'KO/TKO' THEN 1
        WHEN method = 'Submission' THEN 1
        WHEN method = 'TKO - Doctor''s Stoppage' THEN 1
        ELSE 0
    END AS numMethod
    FROM fightsTable
    where winner = fighter
)
"""
finishRateResult = sqldf(query)

In [9]:

""" 
This block is for calculating the striking differential
"""
query = """ 
SELECT fighter, fighterID, eventDate, eventID, fightID, significantStrikesLanded, significantStrikesAbsorbed
FROM(
    SELECT fighterA as fighter, fighter_A_ID as fighterID, fighter_A_sig_strikes as significantStrikesLanded, fighter_B_sig_strikes as significantStrikesAbsorbed, eventDate, eventID, fightID
    FROM fightsTable
    UNION ALL
    SELECT fighterB as fighter, fighter_B_ID as fighterID, fighter_B_sig_strikes as significantStrikesLanded, fighter_A_sig_strikes as significantStrikesAbsorbed, eventDate, eventID, fightID
    FROM fightsTable
)
"""
strikeDifferentialResult = sqldf(query)

strikeDifferentialResult = strikeDifferentialResult.sort_values(by=['fighterID', 'eventDate'])
strikeDifferentialResult['cumulativeStrikesLanded'] = strikeDifferentialResult.groupby('fighterID')['significantStrikesLanded'].cumsum().shift(1)
strikeDifferentialResult['cumulativeStrikesAbsorbed'] = strikeDifferentialResult.groupby('fighterID')['significantStrikesAbsorbed'].cumsum().shift(1)

# Calculate the strikeDifferential
strikeDifferentialResult['strikeDifferential'] = strikeDifferentialResult['cumulativeStrikesLanded'] / strikeDifferentialResult['cumulativeStrikesAbsorbed']



In [10]:
""" 
This code block is dedicated to finding out the takedown differential
"""
query = """ 
SELECT fighter, fighterID, eventDate, eventID, takedownsLanded, takedownsAbsorbed, fightID
FROM(
    SELECT fighterA as fighter, fighter_A_ID as fighterID, fighter_A_takedowns as takedownsLanded, fighter_B_takedowns as takedownsAbsorbed, eventDate, eventID, fightID
    FROM fightsTable
    UNION ALL
    SELECT fighterB as fighter, fighter_B_ID as fighterID, fighter_B_takedowns as takedownsLanded, fighter_A_takedowns as takedownsAbsorbed, eventDate, eventID, fightID
    FROM fightsTable
)
"""
takedownDifferentialResult = sqldf(query)

takedownDifferentialResult = takedownDifferentialResult.sort_values(by=['fighterID', 'eventDate'])
takedownDifferentialResult['cumulativeTakedownsLanded'] = takedownDifferentialResult.groupby('fighterID')['takedownsLanded'].cumsum().shift(1)
takedownDifferentialResult['cumulativeTakedownsAbsorbed'] = takedownDifferentialResult.groupby('fighterID')['takedownsAbsorbed'].cumsum().shift(1)


# Calculate the takedownDifferential
takedownDifferentialResult['takedownDifferential'] = takedownDifferentialResult['cumulativeTakedownsLanded'] / takedownDifferentialResult['cumulativeTakedownsAbsorbed']

In [11]:
###
#UPDATED -> This doesn't really need an update. Thank goodness!
###
""" 
This block will extract the weight class and stance for each fighter
"""
query = """ 
SELECT fighterA as fighter, fighter_A_ID as fighterID, weightClass, fightersTable.stance, eventDate, eventID, fightID
FROM fightsTable LEFT JOIN
fightersTable on fightsTable.fighter_A_ID = fightersTable.fighterID
UNION ALL
SELECT fighterB as fighter, fighter_B_ID as fighterID, weightClass, fightersTable.stance, eventDate, eventID, fightID
FROM fightsTable LEFT JOIN
fightersTable on fightsTable.fighter_B_ID = fightersTable.fighterID
"""
weightClassResult = sqldf(query)

In [12]:
###
#UPDATED -> this is where I noticed the mistake, fixed!
###
""" 
This block will be focused on calculating the winstreak for fighters
"""

def createWinStreak(df: pd.DataFrame) -> pd.DataFrame:

    currentFighter = df['fighter'][0]
    refreshStreak = 0
    for index, row in df.iterrows():

        #handle first row
        if index == 0:
            df.loc[index, 'winStreak'] = refreshStreak
            continue

        #if we hit a new fighter auto insert 0 as the winstreak, update, and move on
        if currentFighter != row['fighter']:
            currentFighter = row['fighter']
            df.loc[index, 'winStreak'] = refreshStreak
            continue

        #catch when fighter lost their last fight, can insert 0 and move on
        if df.loc[(index-1), 'winner'] != df.loc[(index-1), 'fighter']:
            df.loc[index, 'winStreak'] = refreshStreak
            continue
        
        #since we are here it means that we are still on the same fighter and they havent lost
        df.loc[index, 'winStreak'] = df.loc[(index-1), 'winStreak'] + 1

    return df

query = """ 
select *
FROM(
SELECT fighterA as fighter, fighterB as opponent, fighter_A_ID as fighterID, winner, eventDate, eventID, fightID,
CASE
    WHEN fighterA = winner THEN 1
    ELSE 0
end as winResult
FROM fightsTable 
UNION ALL
SELECT fighterB as fighter, fighterA as oponnent, fighter_B_ID as fighterID, winner, eventDate, eventID, fightID,
CASE
    WHEN fighterB = winner THEN 1
    ELSE 0
end as winResult
FROM fightsTable 
)
order by fighterID, eventDate
"""
winstreakResult = sqldf(query)
winstreakResult = createWinStreak(winstreakResult)

In [13]:
""" 
This code block is dedicated towards finding the fighters average control time
"""
query = """ 
SELECT fighterA as fighter, fighter_A_ID as fighterID, fighter_A_control_time as controlTime, `Minutes In Fight`, eventDate, eventID, fightID
FROM fightsTable 
UNION ALL
SELECT fighterB as fighter, fighter_B_ID as fighterID, fighter_B_control_time as controlTime, `Minutes In Fight`, eventDate, eventID, fightID
FROM fightsTable 
"""
averageControlTimeResult = sqldf(query)

averageControlTimeResult[['minutes', 'seconds']] = averageControlTimeResult['controlTime'].str.split(':', expand=True)
averageControlTimeResult['minutes'] = averageControlTimeResult['minutes'].astype(int)
averageControlTimeResult['seconds'] = averageControlTimeResult['seconds'].astype(int)
averageControlTimeResult['Control Time In Minutes'] = averageControlTimeResult['minutes'] + averageControlTimeResult['seconds'] / 60
averageControlTimeResult.drop(['minutes', 'seconds'], axis=1, inplace=True)
averageControlTimeResult['Control Percentage'] = averageControlTimeResult['Control Time In Minutes'] / averageControlTimeResult['Minutes In Fight']

query = """ 
select *, AVG(`Control Percentage`) OVER (PARTITION BY fighterID ORDER BY eventDate ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) as `Average Control Time`
from averageControlTimeResult
"""
averageControlPercentageResult = sqldf(query)

In [14]:
#create final df and start to join everything together
finalStats = averageControlPercentageResult.loc[:, ['fighter', 'fighterID', 'Average Control Time', 'eventID', 'fightID']]

def joinTogether(orgDF, joiningDF, specialColumn):
    # Enclose the specialColumn in backticks if it contains spaces or special characters
    specialColumn = f'`{specialColumn}`' if ' ' in specialColumn or not specialColumn.isidentifier() else specialColumn
    
    query = f""" 
    SELECT odf.*, jdf.{specialColumn}
    FROM {orgDF} as odf LEFT JOIN
    {joiningDF} as jdf ON odf.eventID = jdf.eventID
    AND odf.fighterID = jdf.fighterID
    """
    return sqldf(query)

testDF = joinTogether('finalStats', 'experienceResult', 'Number of Fights')
testDF = joinTogether('testDF', 'winPercentageResult', 'winPercentage')
testDF = joinTogether('testDF', 'averageFightTimeResult', 'Average Fight Time')
testDF = joinTogether('testDF', 'ageResult', 'Age')
testDF = joinTogether('testDF', 'finishRateResult', 'Finish Rate')
testDF = joinTogether('testDF', 'strikeDifferentialResult', 'strikeDifferential')
testDF = joinTogether('testDF', 'takedownDifferentialResult', 'takedownDifferential')
testDF = joinTogether('testDF', 'weightClassResult', 'weightClass')
testDF = joinTogether('testDF', 'winstreakResult', 'winStreak')
testDF = joinTogether('testDF', 'averageControlPercentageResult', 'Control Percentage')
testDF


,fighter,fighterID,Average Control Time,eventID,fightID,Number of Fights,winPercentage,Average Fight Time,Age,Finish Rate,strikeDifferential,takedownDifferential,weightClass,winStreak,Control Percentage
0,Danny Abbadi,2,NaN,621,7094,0,NaN,NaN,22.976044,NaN,NaN,NaN,Middleweight,0.0,0.000000
1,Danny Abbadi,2,0.000000,616,7051,1,0.000000,2.933333,23.225188,NaN,0.142857,0.000000,Lightweight,0.0,0.122222
2,David Abbott,4,NaN,647,7303,0,NaN,NaN,NaN,NaN,0.746835,0.000000,Heavyweight,0.0,0.826087
3,David Abbott,4,0.826087,645,7287,1,0.000000,0.766667,NaN,NaN,inf,NaN,Heavyweight,0.0,0.008403
4,David Abbott,4,0.417245,643,7271,2,0.000000,1.375000,NaN,NaN,1.000000,0.000000,Heavyweight,0.0,0.626866
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14867,Cat Zingano,4179,0.427368,330,3919,3,0.666667,8.166667,34.023272,NaN,1.666667,1.000000,Women's Bantamweight,0.0,0.605556
14868,Cat Zingano,4179,0.471915,261,3110,4,0.500000,9.875000,35.671458,NaN,1.508772,1.166667,Women's Bantamweight,0.0,0.015556
14869,Cat Zingano,4179,0.380643,248,2956,5,0.400000,10.900000,36.035592,1.0,1.479592,0.875000,Women's Bantamweight,0.0,0.593333
14870,Cat Zingano,4179,0.416091,230,2731,6,0.500000,11.583333,36.495551,NaN,1.589286,1.625000,Women's Featherweight,1.0,0.000000


In [15]:
testDF['row_number'] = testDF.groupby('fightID').cumcount() + 1

# Define the SQL query to separate fighters and their stats
query = """
SELECT 
    fightID,
    MAX(CASE WHEN row_number = 1 THEN fighter ELSE NULL END) AS fighterA,
    MAX(CASE WHEN row_number = 2 THEN fighter ELSE NULL END) AS fighterB,
    MAX(CASE WHEN row_number = 1 THEN fighterID ELSE NULL END) AS fighterIDA,
    MAX(CASE WHEN row_number = 2 THEN fighterID ELSE NULL END) AS fighterIDB,
    MAX(CASE WHEN row_number = 1 THEN `Average Control Time` ELSE NULL END) AS averageControlTimeA,
    MAX(CASE WHEN row_number = 2 THEN `Average Control Time` ELSE NULL END) AS averageControlTimeB,
    MAX(CASE WHEN row_number = 1 THEN winPercentage ELSE NULL END) AS winPercentageA,
    MAX(CASE WHEN row_number = 2 THEN winPercentage ELSE NULL END) AS winPercentageB,
    MAX(CASE WHEN row_number = 1 THEN `Average Fight Time` ELSE NULL END) AS averageFightTimeA,
    MAX(CASE WHEN row_number = 2 THEN `Average Fight Time` ELSE NULL END) AS averageFightTimeB,
    MAX(CASE WHEN row_number = 1 THEN Age ELSE NULL END) AS ageA,
    MAX(CASE WHEN row_number = 2 THEN Age ELSE NULL END) AS ageB,
    MAX(CASE WHEN row_number = 1 THEN `Finish Rate` ELSE NULL END) AS finishRateA,
    MAX(CASE WHEN row_number = 2 THEN `Finish Rate` ELSE NULL END) AS finishRateB,
    MAX(CASE WHEN row_number = 1 THEN strikeDifferential ELSE NULL END) AS strikeDifferentialA,
    MAX(CASE WHEN row_number = 2 THEN strikeDifferential ELSE NULL END) AS strikeDifferentialB,
    MAX(CASE WHEN row_number = 1 THEN takedownDifferential ELSE NULL END) AS takedownDifferentialA,
    MAX(CASE WHEN row_number = 2 THEN takedownDifferential ELSE NULL END) AS takedownDifferentialB,
    MAX(CASE WHEN row_number = 1 THEN weightClass ELSE NULL END) AS weightClassA,
    MAX(CASE WHEN row_number = 2 THEN weightClass ELSE NULL END) AS weightClassB,
    MAX(CASE WHEN row_number = 1 THEN winStreak ELSE NULL END) AS winStreakA,
    MAX(CASE WHEN row_number = 2 THEN winStreak ELSE NULL END) AS winStreakB,
    MAX(CASE WHEN row_number = 1 THEN `Control Percentage` ELSE NULL END) AS controlPercentageA,
    MAX(CASE WHEN row_number = 2 THEN `Control Percentage` ELSE NULL END) AS controlPercentageB,
    MAX(eventID) AS eventID,
    COUNT(*) AS numberOfFights
FROM testDF
GROUP BY fightID
"""

# Execute the SQL query using pandasql
result_df = sqldf(query)

# Display the result
result_df


,fightID,fighterA,fighterB,fighterIDA,fighterIDB,averageControlTimeA,averageControlTimeB,winPercentageA,winPercentageB,averageFightTimeA,...,takedownDifferentialA,takedownDifferentialB,weightClassA,weightClassB,winStreakA,winStreakB,controlPercentageA,controlPercentageB,eventID,numberOfFights
0,1,Edson Barboza,Lerone Murphy,252,2602,0.057191,0.294045,0.620690,0.833333,11.751149,...,0.458333,0.352941,Featherweight,Featherweight,2.0,5.0,0.011333,0.212667,2,2
1,2,Carlston Harris,Khaos Williams,1501,4056,0.254354,0.111575,0.800000,0.714286,7.300000,...,1.250000,0.000000,Welterweight,Welterweight,2.0,1.0,0.000000,0.011111,2,2
2,3,Ramiz Brahimaj,Themba Gorimbo,425,1364,0.419089,0.402839,0.500000,0.666667,7.908333,...,0.800000,2.500000,Welterweight,Welterweight,1.0,2.0,0.248889,0.681111,2,2
3,4,Vinicius Salvador,Adrian Yanez,3289,4116,0.012222,0.012011,0.000000,0.714286,15.000000,...,0.333333,NaN,Bantamweight,Bantamweight,0.0,0.0,0.000000,0.101796,2,2
4,5,Angela Hill,Luana Pinheiro,1582,2967,0.142360,0.132932,0.458333,0.750000,14.522917,...,0.629630,8.000000,Women's Strawweight,Women's Strawweight,1.0,0.0,0.336957,0.007246,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7431,29733,Grant Dawson,Joe Solecki,847,3564,0.581434,0.611613,0.900000,0.714286,11.588333,...,14.500000,13.000000,Lightweight,Lightweight,0.0,0.0,0.917778,0.015556,696,2
7432,29734,Jake Matthews,Phil Rowe,2340,3233,0.241418,0.164822,0.631579,0.600000,11.389474,...,2.300000,0.500000,Welterweight,Welterweight,0.0,0.0,0.204444,0.122222,696,2
7433,29735,Mickey Gall,Bassil Hafez,1239,1456,0.266785,0.453333,0.545455,0.000000,7.877273,...,1.000000,3.000000,Welterweight,Welterweight,0.0,0.0,0.008889,0.080000,696,2
7434,29736,Joselyne Edwards,Ailin Perez,1018,2917,0.141746,0.501077,0.571429,0.666667,15.000000,...,0.538462,6.500000,Women's Bantamweight,Women's Bantamweight,0.0,2.0,0.345556,0.286667,696,2


In [20]:
filteredDf = result_df[(result_df['fighterA'] == 'Dustin Poirier') | (result_df['fighterB'] == 'Dustin Poirier')]
filteredDf

,fightID,fighterA,fighterB,fighterIDA,fighterIDB,averageControlTimeA,averageControlTimeB,winPercentageA,winPercentageB,averageFightTimeA,...,takedownDifferentialA,takedownDifferentialB,weightClassA,weightClassB,winStreakA,winStreakB,controlPercentageA,controlPercentageB,eventID,numberOfFights
115,116,Dustin Poirier,Benoit Saint Denis,2973,3266,0.215974,0.500011,0.724138,0.833333,10.102299,...,1.076923,3.250000,Lightweight,Lightweight,0.0,5.0,0.068584,0.648230,11,2
414,415,Justin Gaethje,Dustin Poirier,1231,2973,0.059453,0.223687,0.636364,0.750000,10.106061,...,0.200000,1.076923,Lightweight,Lightweight,1.0,1.0,0.000000,0.000000,36,2
784,785,Michael Chandler,Dustin Poirier,630,2973,0.248545,0.229554,0.500000,0.740741,7.025000,...,2.000000,1.217391,Lightweight,Lightweight,1.0,0.0,0.470833,0.065278,66,2
1256,1257,Charles Oliveira,Dustin Poirier,2756,2973,0.236483,0.236350,0.678571,0.769231,6.891071,...,1.888889,1.217391,Lightweight,Lightweight,9.0,3.0,0.515106,0.052870,105,2
1482,1483,Conor McGregor,Dustin Poirier,2382,2973,0.157772,0.219404,0.769231,0.760000,8.273077,...,0.500000,1.173913,Lightweight,Lightweight,0.0,2.0,0.000000,0.660000,124,2
1717,1718,Conor McGregor,Dustin Poirier,2382,2973,0.158567,0.219143,0.833333,0.750000,8.334722,...,0.555556,1.130435,Lightweight,Lightweight,1.0,1.0,0.148230,0.225664,144,2
2026,2027,Dan Hooker,Dustin Poirier,1623,2973,0.152472,0.223772,0.714286,0.739130,9.875000,...,0.333333,1.315789,Lightweight,Lightweight,3.0,0.0,0.241333,0.112667,172,2
2369,2370,Khabib Nurmagomedov,Dustin Poirier,2720,2973,0.529770,0.233944,1.000000,0.772727,13.931818,...,26.000000,2.083333,Lightweight,Lightweight,11.0,4.0,0.732782,0.000000,201,2
2587,2588,Max Holloway,Dustin Poirier,1603,2973,0.070068,0.238259,0.842105,0.761905,12.409649,...,0.277778,2.000000,Lightweight,Lightweight,13.0,3.0,0.004000,0.143333,219,2
2924,2925,Eddie Alvarez,Dustin Poirier,105,2973,0.167341,0.249713,0.571429,0.750000,11.414286,...,inf,2.000000,Lightweight,Lightweight,1.0,2.0,0.271560,0.009174,246,2
